In [1]:
!pip install -q --upgrade diffusers transformers accelerate peft datasets torchao

import torch
import matplotlib.pyplot as plt

from diffusers import  StableDiffusionPipeline
from PIL import Image

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("hugging-face")

login(token)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
lora_id = "homerio/estilo_arquitetura_modernista_brasilia_v2"
device = "cuda"

In [3]:
prompts = ["estilo_arquitetura_modernista_brasilia, o Congresso NAcional sob um ceu azul do cerrado",
           "estilo_arquitetura_modernista_brasilia, o Teatro Nacional com sua faichada geometrica de quardados em concreto",
           "estilo_arquitetura_modernista_brasilia, a Catedral Metropolina sob um ceu chuvoso de inverno no cerrado",
           "estilo_arquitetura_modernista_brasilia, uma quadra residencial com seus predios de 4 andares e pilotis livres",
           "estilo_arquitetura_modernista_brasilia, o Palacio do Planalto numa vista alta com a ponte JK ao fundo",
           "estilo_arquitetura_modernista_brasilia, uma igreja catolica com a arquitetura geometrica sob um ceu limpo do cerrado"]

seed = 777

In [4]:
def gerar_comparativo(prompts, seed, model_id, lora_id=None):
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(device)

    if lora_id is not None:
        print(f"Carregando LoRA: {lora_id}")
        pipe.load_lora_weights(lora_id)
    else:
        print("Nenhum LoRA fornecido, usando modelo base puro.")

    imagens = []

    for prompt in prompts:
        generator = torch.Generator(device).manual_seed(seed)
        image = pipe(prompt, generator=generator).images[0]
        imagens.append(image)

    del pipe
    torch.cuda.empty_cache()

    return imagens



In [ ]:
print  ("Gerando imgs do modelo base ...")

col_base = gerar_comparativo(prompts, seed, model_id, None)

print  ("Gerando imgs do modelo Lora ...")

col_lora = gerar_comparativo(prompts, seed, model_id, lora_id)



Gerando imgs do modelo base ...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

...

Nenhum LoRA fornecido, usando modelo base puro.


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(len(prompts), 2, figsize=(12, 24))

cols = ["Modelo Base (SD 1.5)", "Modelo Fine-Tunado LoRA Estilo Brasilia "]

for i in range(len(prompts)):
  axes[i, 0].imshow(col_base[i])
  # axes[i, 0].set_title(cols[0])
  axes[i, 0].axis("off")

  axes[i, 1].imshow(col_lora[i])
  # axes[i, 1].set_title(cols[1])
  axes[i, 1].axis("off")

  if i == 0:
    axes[i, 0].set_title(cols, fontsize = 16)
    axes[i, 1].set_title(cols[4], fontsize = 16)

plt.tight_layout()
plt.savefig("comparativo_lora.png")
plt.show()

In [ ]:
# !pip -q install torchmetrics 
# import torch 
# from torchmetrics.multimodal.clip_score import CLIPScore 
  
# metrica = CLIPScore(model_name_or_path="openai/clip-vit-base patch16").to("cuda") 
# score = metrica(tensor_da_imagem, "estilo_cordel, um vaqueiro tocando viola") 
# print(f"CLIPScore: {score.item():.2f}") 